In [4]:
import sys
sys.path.append('.')
import numpy as np
from astropy.io import fits
from astropy.wcs import WCS
import math
import pandas as pd
import os

In [9]:
import os
import pandas as pd

# Define column names based on the header in the SRL file
column_names = [
    "Source_id", "Isl_id", "RA", "E_RA", "DEC", "E_DEC", "Total_flux", "E_Total_flux",
    "Peak_flux", "E_Peak_flux", "RA_max", "E_RA_max", "DEC_max", "E_DEC_max", "Maj", 
    "E_Maj", "Min", "E_Min", "PA", "E_PA", "Maj_img_plane", "E_Maj_img_plane", 
    "Min_img_plane", "E_Min_img_plane", "PA_img_plane", "E_PA_img_plane", "DC_Maj", 
    "E_DC_Maj", "DC_Min", "E_DC_Min", "DC_PA", "E_DC_PA", "DC_Maj_img_plane", 
    "E_DC_Maj_img_plane", "DC_Min_img_plane", "E_DC_Min_img_plane", "DC_PA_img_plane", 
    "E_DC_PA_img_plane", "Isl_Total_flux", "E_Isl_Total_flux", "Isl_rms", "Isl_mean", 
    "Resid_Isl_rms", "Resid_Isl_mean", "S_Code"
]

spw = [2, 3, 4, 5, 6, 8, 15, 16, 17]
dataframes = []

for s in spw:
    # Define SRL file path for each SPW
    path = f'../paper1/spw/32_pb/spw{s}-2.5arcsec-nit5000-awproject.pbcor.pybdsf.srl'
    
    if os.path.exists(path):
        try:
            # Read SRL file into a DataFrame
            df = pd.read_csv(path, delim_whitespace=True, comment='#', names=column_names)
            dataframes.append(df)
        except Exception as e:
            print(f"Error reading {path}: {e}")
    else:
        print(f"File not found: {path}")

# Combine all DataFrames into one for finding unique sources
if dataframes:
    all_data = pd.concat(dataframes, ignore_index=True)
    
    # Find unique RA and DEC values
    unique_sources = all_data.drop_duplicates(subset=['RA', 'DEC'])[['RA', 'DEC']]

    # Save unique RA and DEC to a CSV file
    unique_sources_csv = '../paper2/32/unique_sources.csv'
    unique_sources.to_csv(unique_sources_csv, index=False)
    print(f"Unique RA and DEC values saved at {unique_sources_csv}")

    print(f"Total unique sources (RA and DEC) found: {len(unique_sources)}")
else:
    print("No data found to process.")

Unique RA and DEC values saved at ../paper2/32/unique_sources.csv
Total unique sources (RA and DEC) found: 2123


In [10]:
unique_sources

,RA,DEC
0,54.622813,31.480799
1,54.500236,31.103612
2,54.464928,30.934108
3,54.482100,30.956292
4,54.456879,31.254856
...,...,...
2118,51.614963,30.273440
2119,51.589144,30.724840
2120,51.578286,31.455469
2121,51.585872,30.783017


In [2]:
dx = 4
dy = 4
deltax = 10
deltay = 10
deltaxy = 7

Stokes = ['I', 'Q', 'U']

In [ ]:
manual_setting = 1

def ra_dec_to_degrees(ra, dec):
    ra = ra.split(':')
    ra_hours, ra_minutes, ra_seconds = int(ra[0]), int(ra[1]), int(ra[2])

    dec = dec.split(':')
    dec_degrees, dec_minutes, dec_seconds = int(dec[0]), int(dec[1]), int(dec[2])
    ra_degrees = (ra_hours + ra_minutes / 60 + ra_seconds / 3600) * 15
    sign = 1 if dec_degrees >= 0 else -1
    dec_degrees = abs(dec_degrees) + dec_minutes / 60 + dec_seconds / 3600
    dec_degrees *= sign

    return ra_degrees, dec_degrees

# Manual setting RA/Dec coordinates
if manual_setting == 1:
    ra_list = ['3:31:54', '3:34:16', '3:33:21', '3:35:05', '3:31:19', '3:30:09', '3:28:40', '3:32:41', '3:34:01']
    dec_list = ['31:04:39', '31:12:10', '30:55:28', '30:47:04', '30:47:23', '30:32:53', '30:49:56', '30:20:11', '31:18:40']
    RA0, DEC0 = [], []
    for ra, dec in zip(ra_list, dec_list):
        radec = ra_dec_to_degrees(ra, dec)
        RA0.append(radec[0])
        DEC0.append(radec[1])
else:
    imagename = f"{threedigits}-mosaic-fieldALL-Stokes{str(Stoke)}-2.5arc-{str(nit)}-{str(thresh)}-spwALL-pb{str(pblim)}-cyclenit500"
    catalog_name = f"{path}/Images/{imagename}.image.pybdsf.srl"
    df = pd.read_csv(catalog_name, delim_whitespace=True)
    RA0 = df['RA'].tolist()
    DEC0 = df['DEC'].tolist()

# Process each Stokes file
for stokes in Stokes:
    fits_file = f'../paper1/32/Stokes{stokes}.fits'
    hdulist = fits.open(fits_file)
    img = hdulist[0].data
    wcs = WCS(hdulist[0].header)
    stok = {'I': 1, 'Q': 2, 'U': 3}[stokes]

    for ip, (ra, dec) in enumerate(zip(RA0, DEC0)):
        sky1 = [[ra, dec, 1.47339395E9, 0]]
        pixcrd2 = wcs.all_world2pix(sky1, 0)
        x0, y0 = int(pixcrd2[0][0] + 0.5), int(pixcrd2[0][1] + 0.5)

        sourcename_RADEC = f"H{ra:+07.3f}{dec:+06.3f}"
        spectrum_file = f"{sourcename_RADEC}_{dx}_{dy}_{stokes}.pro"
        OUT = open(f'../paper1/32/RMsyn/{spectrum_file}', "w")

        print("Pixel coordinates: ", x0, y0, sourcename_RADEC, spectrum_file)
        print(f"## RA DEC: {ra} {dec}", file=OUT)
        print(f"## Pixel coordinates: {x0} {y0}  Box: {dx} {dy}", file=OUT)
        print(f"## Source name: {sourcename_RADEC} Stokes {stokes}", file=OUT)
        print(f"## Spectrum file: {spectrum_file}", file=OUT)

        box_spectrum = img[:, (y0 - dy):(y0 + dy), (x0 - dx):(x0 + dx)]
        mean_spectrum = img[:, x0, y0]
        freq = np.copy(mean_spectrum)
        flag = np.zeros_like(mean_spectrum)

        for ispec in range(len(mean_spectrum)):
            sky = wcs.all_pix2world([[x0, y0, ispec, stok]], 0)
            freq[ispec] = sky[0][2]
            mean_spectrum[ispec] = np.average(box_spectrum[ispec, :, :])

            backgrounds = [img[ispec, x0, y0 + dy], img[ispec, x0, y0 - dy], img[ispec, x0 + deltax, y0],
                           img[ispec, x0 - deltax, y0], img[ispec, x0 + deltax, y0 + dy], img[ispec, x0 + deltax, y0 - dy],
                           img[ispec, x0 - deltax, y0 - dy], img[ispec, x0 - deltax, y0 + dy]]
            diff = mean_spectrum[ispec] - np.array(backgrounds)
            mean_spectrum_reduced = np.median(diff)

            if mean_spectrum[ispec] > 1.E10:
                mean_spectrum[ispec] = 0
                flag[ispec] = 1.0

            print(f"{freq[ispec]:15.12e}  {mean_spectrum_reduced:15.8e}  {int(flag[ispec] + 0.5)}", file=OUT)
        OUT.close()

    rms = np.std(mean_spectrum)
    if rms > 0.002:
        print(sourcename_RADEC, rms)
